# LeetCode 1102 — Path With Maximum Minimum Value bằng Python

Link đề: https://leetcode.com/problems/path-with-maximum-minimum-value/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, DFS, BFS, Union Find, Heap (Priority Queue), Matrix

> **Premium:** đây là bài **Premium** (🔒). Đề trong notebook được viết lại theo hiểu biết và theo bản mô tả trong kho tham khảo; nếu bạn có tài khoản Premium hãy **đối chiếu lại** đề gốc (đặc biệt: điểm của đường đi là **giá trị nhỏ nhất** trên đường, **tính cả** ô đầu và ô cuối; được đi **bốn hướng**; có thể quay lại ô cũ nhưng điều đó không có ích).

Skeleton LeetCode:

```python
class Solution:
    def maximumMinimumPath(self, grid: List[List[int]]) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất**: điểm lớn nhất trong mọi đường đi từ `(0, 0)` đến `(m-1, n-1)` theo `4` hướng, trong đó **điểm của một đường = giá trị nhỏ nhất** trên đường. Điểm cốt lõi: bài toán **cực đại của cực tiểu (maximin / bottleneck path)** — giải bằng **tìm kiếm nhị phân trên đáp án + BFS**, **hàng đợi ưu tiên kiểu Dijkstra**, hoặc **Union-Find kích hoạt ô theo giá trị giảm dần**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lưới `m × n`, đi theo `4` hướng từ góc trên trái đến góc dưới phải; điểm của đường là **giá trị nhỏ nhất** trên đường; tối đa hoá điểm.
2. Nắm công cụ: BFS/DFS trên lưới, tìm kiếm nhị phân trên đáp án, hàng đợi ưu tiên (max-heap) kiểu Dijkstra, Union-Find, thuật toán lan truyền (relaxation).
3. Đi từ **vét cạn mọi đường đơn** → **nhị phân + BFS** → **max-heap kiểu Dijkstra** → **Union-Find theo giá trị giảm dần** → **lan truyền `best[r][c]`**.
4. Hiểu mấu chốt: "có đường với mọi ô `≥ v`" **đơn điệu** theo `v`; đáp án là `v` lớn nhất để hai góc **liên thông** trong đồ thị các ô có giá trị `≥ v`.
5. Tránh lỗi kinh điển: chỉ đi phải/xuống (DP cổ điển), tham lam theo ô lớn nhất, quên ô đầu/ô cuối trong điểm, so sánh `>` thay vì `≥`, dùng min-heap, lấy `min` của hai góc.

## 2. Đề bài & ví dụ

Cho ma trận số nguyên `grid` cỡ `m × n`. Trả về **điểm lớn nhất** của một đường đi từ `(0, 0)` đến `(m - 1, n - 1)` di chuyển theo `4` hướng chính (lên, xuống, trái, phải).

**Điểm** của một đường đi là **giá trị nhỏ nhất** trong đường đó (tính cả ô đầu và ô cuối). Ví dụ điểm của đường `8 → 4 → 5 → 9` là `4`.

**Ví dụ 1:** `grid = [[5,4,5],[1,2,6],[7,4,6]]` → `4` (đường `5 → 4 → 5 → 6 → 6`).

**Ví dụ 2:** `grid = [[2,2,1,2,2,2],[1,2,2,2,1,2]]` → `2`.

**Ví dụ 3:** `grid = [[3,4,6,3,4],[0,2,1,1,7],[8,8,3,2,7],[3,2,4,9,8],[4,1,2,0,0],[4,6,5,4,3]]` → `3`.

**Ràng buộc:** `1 <= m, n <= 100`; `0 <= grid[i][j] <= 10^9`.

In [1]:
import heapq
import itertools
import random
import time
from collections import deque
from typing import List

examples = [
    ([[5, 4, 5], [1, 2, 6], [7, 4, 6]], 4),
    ([[2, 2, 1, 2, 2, 2], [1, 2, 2, 2, 1, 2]], 2),
    ([[3, 4, 6, 3, 4], [0, 2, 1, 1, 7], [8, 8, 3, 2, 7], [3, 2, 4, 9, 8], [4, 1, 2, 0, 0], [4, 6, 5, 4, 3]], 3),
]
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]
for g, want in examples:
    print(f"grid {len(g)} × {len(g[0])} -> {want}")

grid 3 × 3 -> 4
grid 2 × 6 -> 2
grid 6 × 5 -> 3


## 3. Kiến thức nền

### 3.1 Điểm của một đường đi

Với đường đi `P = (c_0 = (0,0), c_1, …, c_k = (m-1,n-1))` các ô liên tiếp kề nhau theo `4` hướng, `score(P) = min_i grid[c_i]`. Đáp án `= max_P score(P)`. Quay lại ô cũ không bao giờ làm điểm tốt hơn (thêm ô chỉ có thể **giảm** min), nên chỉ cần xét **đường đơn** (không lặp ô).

In [2]:
def score(grid, path):
    return min(grid[r][c] for r, c in path)


g = examples[0][0]
p = [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]
print(score(g, p), score(g, [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2)]))
assert score(g, p) == 4 and score(g, [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2)]) == 1

4 1


### 3.2 Đường với mọi ô `≥ v`

Gọi `reach(v)` = "tồn tại đường từ `(0,0)` đến `(m-1,n-1)` mà **mọi ô** có giá trị `≥ v`". Một đường có `score ≥ v` ⇔ mọi ô `≥ v`. Vậy `đáp án = max{v : reach(v)}`.

**Đơn điệu:** nếu `reach(v)` đúng thì `reach(v')` đúng với mọi `v' ≤ v` (cùng đường đó). Do đó ta có thể tìm `v` lớn nhất bằng tìm kiếm nhị phân (trên các giá trị có trong lưới).

### 3.3 Liên thông trong đồ thị các ô `≥ v`

`reach(v)` ⇔ `(0,0)` và `(m-1,n-1)` thuộc cùng một thành phần liên thông của đồ thị các ô có giá trị `≥ v` (cạnh nối ô kề nhau). Khi `v` giảm dần, ngày càng nhiều ô được thêm vào: Union-Find phù hợp.

In [3]:
def reachable(grid, v):
    """BFS: có đường từ (0,0) đến (m-1,n-1) chỉ qua các ô >= v không?"""
    m, n = len(grid), len(grid[0])
    if grid[0][0] < v or grid[m - 1][n - 1] < v:
        return False
    seen = [[False] * n for _ in range(m)]
    seen[0][0] = True
    dq = deque([(0, 0)])
    while dq:
        r, c = dq.popleft()
        if (r, c) == (m - 1, n - 1):
            return True
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc] and grid[nr][nc] >= v:
                seen[nr][nc] = True
                dq.append((nr, nc))
    return False


g = examples[0][0]
print([reachable(g, v) for v in range(0, 8)])
assert [reachable(g, v) for v in range(0, 8)] == [True, True, True, True, True, False, False, False]

[True, True, True, True, True, False, False, False]


## 4. Ý tưởng nền tảng / trực giác

Hãy nghĩ "mực nước": chỉ những ô cao ít nhất `v` là đi được. Nâng `v` dần lên; đến một mức nào đó hai góc bị tách rời. Mức `v` cao nhất mà hai góc còn thông nhau chính là điểm tốt nhất. Đường đi "mắt xích yếu nhất" là ô thấp nhất; ta muốn **mắt xích yếu nhất cũng càng khoẻ càng tốt**.

## 5. Brute force — thử mọi đường đơn

**Ý tưởng:** DFS liệt kê mọi **đường đơn** (không lặp ô) từ `(0,0)` đến `(m-1,n-1)`, tính `score` mỗi đường, lấy lớn nhất. Đúng theo định nghĩa; số đường tăng rất nhanh (lưới `4 × 4`: `184` đường; lưới lớn không khả thi), dùng làm **oracle** cho lưới nhỏ.

In [4]:
def max_min_brute(grid, dirs=DIRS):
    m, n = len(grid), len(grid[0])
    best = [-1]
    seen = [[False] * n for _ in range(m)]

    def dfs(r, c, cur_min):
        cur_min = min(cur_min, grid[r][c])
        if (r, c) == (m - 1, n - 1):
            best[0] = max(best[0], cur_min)
            return
        seen[r][c] = True
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc]:
                dfs(nr, nc, cur_min)
        seen[r][c] = False

    dfs(0, 0, float("inf"))
    return best[0]


assert max_min_brute([[5, 4, 5], [1, 2, 6], [7, 4, 6]]) == 4
print([max_min_brute(g) for g, _ in examples[:1]])

[4]


In [5]:
# Code nộp LeetCode cho Brute force (đúng nhưng số đường đơn tăng mũ: chỉ phù hợp lưới nhỏ, KHÔNG qua test lớn)
class SolutionBrute:
    def maximumMinimumPath(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        best = [-1]
        seen = [[False] * n for _ in range(m)]

        def dfs(r, c, cur):
            cur = min(cur, grid[r][c])
            if r == m - 1 and c == n - 1:
                best[0] = max(best[0], cur)
                return
            seen[r][c] = True
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc]:
                    dfs(nr, nc, cur)
            seen[r][c] = False

        dfs(0, 0, float("inf"))
        return best[0]

**Độ phức tạp:** số đường đơn (mũ theo `m · n`); `O(m · n)` bộ nhớ đệ quy.

**Nút thắt:** ta không cần biết **đường nào**, chỉ cần biết **mức `v` cao nhất** để hai góc còn thông.

## 6. Bước đột phá

Hai cách tương đương: (i) **nhị phân theo `v`** với BFS kiểm tra `reach(v)`; (ii) **mở rộng theo giá trị giảm dần** (max-heap hoặc Union-Find) cho đến khi hai góc nối được: mức `v` lúc đó là đáp án.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** nhị phân + BFS, **Cách 3** max-heap kiểu Dijkstra (đáp án chính), **Cách 4** Union-Find theo giá trị giảm dần, **Cách 5** lan truyền `best[r][c]`.

### Cách 2 — Nhị phân trên đáp án + BFS

**Giải thích:** sắp xếp các giá trị khác nhau trong lưới; nhị phân tìm giá trị lớn nhất `v` với `reach(v)` (đơn điệu theo mục 3.2). `reach` dùng BFS `O(m n)`.

In [6]:
def max_min_bsearch(grid):
    vals = sorted({v for row in grid for v in row})
    lo, hi = 0, len(vals) - 1                              # vals[0] luôn thoả vì ô nào cũng >= giá trị nhỏ nhất
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if reachable(grid, vals[mid]):
            lo = mid
        else:
            hi = mid - 1
    return vals[lo]


for g, want in examples:
    assert max_min_bsearch(g) == want
print([max_min_bsearch(g) for g, _ in examples])

[4, 2, 3]


In [7]:
# Code nộp LeetCode cho Cách 2
class SolutionBSearch:
    def maximumMinimumPath(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])

        def ok(v):
            if grid[0][0] < v or grid[m - 1][n - 1] < v:
                return False
            seen = [[False] * n for _ in range(m)]
            seen[0][0] = True
            stack = [(0, 0)]
            while stack:
                r, c = stack.pop()
                if r == m - 1 and c == n - 1:
                    return True
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc] and grid[nr][nc] >= v:
                        seen[nr][nc] = True
                        stack.append((nr, nc))
            return False

        vals = sorted({v for row in grid for v in row})
        lo, hi = 0, len(vals) - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if ok(vals[mid]):
                lo = mid
            else:
                hi = mid - 1
        return vals[lo]

**Độ phức tạp:** `O(m n log(m n))` thời gian (`log` số giá trị khác nhau, mỗi lần BFS `O(m n)`), `O(m n)` bộ nhớ.

### Cách 3 — Max-heap kiểu Dijkstra (đáp án chính)

**Ý tưởng:** lan truyền từ `(0,0)`, mỗi lần lấy ra ô **có giá trị lớn nhất** trong biên (max-heap), cập nhật `ans = min(ans, giá trị ô vừa lấy)`; khi lấy ra ô đích thì dừng và trả `ans`. Cách này giống Dijkstra với phép "cộng" là `min` và phép so sánh là `max` (đường đi "cổ chai").

In [8]:
def max_min_heap(grid):
    m, n = len(grid), len(grid[0])
    heap = [(-grid[0][0], 0, 0)]                           # max-heap bằng cách đảo dấu
    seen = [[False] * n for _ in range(m)]
    seen[0][0] = True
    ans = grid[0][0]
    while heap:
        neg, r, c = heapq.heappop(heap)
        ans = min(ans, -neg)                               # điểm hiện tại = min giá trị các ô đã lấy ra
        if (r, c) == (m - 1, n - 1):
            return ans
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc]:
                seen[nr][nc] = True
                heapq.heappush(heap, (-grid[nr][nc], nr, nc))
    return ans


for g, want in examples:
    assert max_min_heap(g) == want
print([max_min_heap(g) for g, _ in examples])

[4, 2, 3]


In [9]:
# Code nộp LeetCode cho Cách 3
class SolutionHeap:
    def maximumMinimumPath(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        heap = [(-grid[0][0], 0, 0)]
        seen = [[False] * n for _ in range(m)]
        seen[0][0] = True
        ans = grid[0][0]
        while heap:
            neg, r, c = heapq.heappop(heap)
            ans = min(ans, -neg)
            if r == m - 1 and c == n - 1:
                return ans
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc]:
                    seen[nr][nc] = True
                    heapq.heappush(heap, (-grid[nr][nc], nr, nc))
        return ans

**Độ phức tạp:** `O(m n log(m n))` thời gian, `O(m n)` bộ nhớ.

### Cách 4 — Union-Find, kích hoạt ô theo giá trị giảm dần

**Ý tưởng:** sắp xếp các ô theo giá trị **giảm dần**. Lần lượt "kích hoạt" từng ô: hợp nhất ô đó với các hàng xóm đã kích hoạt; ngay khi `(0,0)` và `(m-1,n-1)` cùng nhóm (cả hai đã được kích hoạt), giá trị của ô vừa kích hoạt là đáp án.

In [10]:
def max_min_dsu(grid):
    m, n = len(grid), len(grid[0])
    parent = list(range(m * n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    active = [[False] * n for _ in range(m)]
    cells = sorted(((grid[r][c], r, c) for r in range(m) for c in range(n)), reverse=True)
    start, end = 0, (m - 1) * n + (n - 1)
    for v, r, c in cells:
        active[r][c] = True
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and active[nr][nc]:
                parent[find(r * n + c)] = find(nr * n + nc)
        if active[0][0] and active[m - 1][n - 1] and find(start) == find(end):
            return v
    return -1


for g, want in examples:
    assert max_min_dsu(g) == want
print([max_min_dsu(g) for g, _ in examples])

[4, 2, 3]


In [11]:
# Code nộp LeetCode cho Cách 4
class SolutionDSU:
    def maximumMinimumPath(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        parent = list(range(m * n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        active = [[False] * n for _ in range(m)]
        cells = sorted(((grid[r][c], r, c) for r in range(m) for c in range(n)), reverse=True)
        for v, r, c in cells:
            active[r][c] = True
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and active[nr][nc]:
                    parent[find(r * n + c)] = find(nr * n + nc)
            if active[0][0] and active[m - 1][n - 1] and find(0) == find(m * n - 1):
                return v
        return -1

**Độ phức tạp:** `O(m n log(m n))` thời gian (sắp xếp), `O(m n)` bộ nhớ.

### Cách 5 — Lan truyền `best[r][c]`

**Ý tưởng:** `best[r][c]` = điểm tốt nhất của một đường từ `(0,0)` đến `(r,c)`. Khởi tạo `best[0][0] = grid[0][0]`, ô khác `-1`. Mỗi khi ô được cải thiện, đẩy vào hàng đợi và thử cải thiện các ô kề: `cand = min(best[cell], grid[nb])`, nếu `cand > best[nb]` thì cập nhật. Giá trị chỉ tăng và bị chặn nên dừng; đáp án `best[m-1][n-1]`.

In [12]:
def max_min_relax(grid):
    m, n = len(grid), len(grid[0])
    best = [[-1] * n for _ in range(m)]
    best[0][0] = grid[0][0]
    dq = deque([(0, 0)])
    in_q = [[False] * n for _ in range(m)]
    in_q[0][0] = True
    while dq:
        r, c = dq.popleft()
        in_q[r][c] = False
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n:
                cand = min(best[r][c], grid[nr][nc])
                if cand > best[nr][nc]:
                    best[nr][nc] = cand
                    if not in_q[nr][nc]:
                        in_q[nr][nc] = True
                        dq.append((nr, nc))
    return best[m - 1][n - 1]


for g, want in examples:
    assert max_min_relax(g) == want
print([max_min_relax(g) for g, _ in examples])

[4, 2, 3]


In [13]:
# Code nộp LeetCode cho Cách 5
class SolutionRelax:
    def maximumMinimumPath(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        best = [[-1] * n for _ in range(m)]
        best[0][0] = grid[0][0]
        dq = deque([(0, 0)])
        in_q = [[False] * n for _ in range(m)]
        in_q[0][0] = True
        while dq:
            r, c = dq.popleft()
            in_q[r][c] = False
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n:
                    cand = min(best[r][c], grid[nr][nc])
                    if cand > best[nr][nc]:
                        best[nr][nc] = cand
                        if not in_q[nr][nc]:
                            in_q[nr][nc] = True
                            dq.append((nr, nc))
        return best[m - 1][n - 1]

**Độ phức tạp:** trường hợp xấu `O((m n)²)` (kiểu Bellman–Ford/SPFA), thường nhanh hơn nhiều; `O(m n)` bộ nhớ.

## 8. TRACE từng bước

Trace **Cách 3** với `grid = [[5,4,5],[1,2,6],[7,4,6]]`:

| bước | lấy ra (giá trị, ô) | `ans` | biên sau khi đẩy hàng xóm |
|---|---|---|---|
| 0 | khởi tạo `(0,0) = 5` | 5 | |
| 1 | `5 @ (0,0)` | 5 | đẩy `(1,0)=1`, `(0,1)=4` |
| 2 | `4 @ (0,1)` | **4** | đẩy `(0,2)=5`, `(1,1)=2` |
| 3 | `5 @ (0,2)` | 4 | đẩy `(1,2)=6` |
| 4 | `6 @ (1,2)` | 4 | đẩy `(2,2)=6` |
| 5 | `6 @ (2,2)` đích | 4 | trả `4` |

Hàm trace in từng bước:

In [14]:
def trace_heap(grid):
    m, n = len(grid), len(grid[0])
    heap = [(-grid[0][0], 0, 0)]
    seen = [[False] * n for _ in range(m)]
    seen[0][0] = True
    ans = grid[0][0]
    step = 0
    print(f"grid = {grid}")
    while heap:
        neg, r, c = heapq.heappop(heap)
        ans = min(ans, -neg)
        pushed = []
        if (r, c) != (m - 1, n - 1):
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc]:
                    seen[nr][nc] = True
                    heapq.heappush(heap, (-grid[nr][nc], nr, nc))
                    pushed.append(((nr, nc), grid[nr][nc]))
        print(f"  bước {step}: lấy {-neg} @ ({r},{c}), ans = {ans}, đẩy {pushed}")
        step += 1
        if (r, c) == (m - 1, n - 1):
            print("  đích: trả", ans)
            return ans
    return ans


assert trace_heap([[5, 4, 5], [1, 2, 6], [7, 4, 6]]) == 4

grid = [[5, 4, 5], [1, 2, 6], [7, 4, 6]]
  bước 0: lấy 5 @ (0,0), ans = 5, đẩy [((1, 0), 1), ((0, 1), 4)]
  bước 1: lấy 4 @ (0,1), ans = 4, đẩy [((1, 1), 2), ((0, 2), 5)]
  bước 2: lấy 5 @ (0,2), ans = 4, đẩy [((1, 2), 6)]
  bước 3: lấy 6 @ (1,2), ans = 4, đẩy [((2, 2), 6)]
  bước 4: lấy 6 @ (2,2), ans = 4, đẩy []
  đích: trả 4


**Trace nhị phân (Cách 2)** cho cùng lưới: các giá trị khác nhau `1,2,4,5,6,7`; `reach(v)` đúng với `v = 1, 2, 4` và sai với `v = 5` (đường tránh được ô `4`? ô `(0,1)=4` và `(1,1)=2`, `(1,0)=1` đều `< 5` nên từ `(0,0)=5` không đi tiếp được); tìm được `v = 4`.

In [15]:
g = [[5, 4, 5], [1, 2, 6], [7, 4, 6]]
print({v: reachable(g, v) for v in sorted({x for r in g for x in r})})
assert max_min_bsearch(g) == 4

{1: True, 2: True, 4: True, 5: False, 6: False, 7: False}


## 9. Code nộp LeetCode

Chọn **Cách 3** (max-heap kiểu Dijkstra). Giữ nguyên `class Solution`, `maximumMinimumPath(self, grid)`:

In [16]:
class Solution:
    def maximumMinimumPath(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        heap = [(-grid[0][0], 0, 0)]                # max-heap: lấy ô lớn nhất trong biên trước
        seen = [[False] * n for _ in range(m)]
        seen[0][0] = True
        ans = grid[0][0]                            # điểm = min giá trị các ô đã đi qua (gồm ô đầu)
        while heap:
            neg, r, c = heapq.heappop(heap)
            ans = min(ans, -neg)
            if r == m - 1 and c == n - 1:           # lấy ra ô đích: ans chính là điểm tốt nhất
                return ans
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc]:
                    seen[nr][nc] = True
                    heapq.heappush(heap, (-grid[nr][nc], nr, nc))
        return ans


print([Solution().maximumMinimumPath(g) for g, _ in examples])
assert all(Solution().maximumMinimumPath(g) == w for g, w in examples)

[4, 2, 3]


## 10. Vì sao đúng

**(a) Đáp án `= max v` thoả `reach(v)`.** Điểm của đường `P` là `min` giá trị; `score(P) ≥ v ⇔` mọi ô của `P` có giá trị `≥ v`. Vậy tồn tại đường điểm `≥ v` ⇔ `reach(v)`; đáp án là `v` lớn nhất thoả `reach(v)`. Giá trị đó phải là một giá trị có mặt trong lưới (điểm là `min` của các ô). `reach(v)` đúng với `v = min(grid)` (mọi ô thoả, lưới liên thông bằng lưới) nên tập thoả không rỗng.

**(b) Nhị phân.** `reach` đơn điệu (mục 3.2) nên nhị phân trên mảng giá trị đã sắp xếp tìm được `v` lớn nhất thoả. `mid = (lo + hi + 1) // 2` (làm tròn lên) tránh vòng lặp vô hạn khi `lo = mid`.

**(c) Max-heap.** Bất biến: tại mỗi bước, các ô đã lấy ra `S` luôn tạo thành một tập **liên thông chứa `(0,0)`**, và giá trị `ans = min(S)`. Ô được lấy ra tiếp theo là ô **lớn nhất trong biên** của `S`. Gọi `v*` là đáp án. Trước khi `ans` hạ xuống dưới `v*`, thuật toán lấy hết mọi ô thuộc thành phần liên thông của `(0,0)` trong đồ thị các ô `≥ v*` (vì mỗi lần lấy ô lớn nhất của biên; một ô `≥ v*` trong biên luôn được lấy trước mọi ô `< v*`). Vì `reach(v*)`, đích nằm trong thành phần đó nên được lấy ra trước khi `ans` giảm dưới `v*`: `ans ≥ v*` khi lấy đích; và `ans ≤ v*` vì `S` tại thời điểm đó chứa một đường từ `(0,0)` đến đích (trong `S` liên thông) mà mọi ô của đường có giá trị `≥ ans`... ta có `ans = min(S) ≥` điểm của một đường trong `S` `≤ v*`. Suy ra `ans = v*`. (Chi tiết: `S` liên thông chứa hai góc nên chứa một đường nối, đường đó có điểm `≥ min(S) = ans`, do đó `v* ≥ ans`.)

**(d) Union-Find.** Sau khi kích hoạt mọi ô có giá trị `≥ v`, các nhóm là các thành phần liên thông của đồ thị các ô `≥ v`. Lần đầu hai góc cùng nhóm xảy ra khi kích hoạt ô có giá trị `v = v*` (giá trị kích hoạt giảm dần): trước đó `reach(v)` sai với mọi `v > v*`; tại `v*` đúng. Ô vừa kích hoạt khi đó có giá trị `v*` (nếu có nhiều ô cùng giá trị thì đã kích hoạt trong cùng nhóm giá trị, và lần đầu nối được xảy ra ở ô nào đó có giá trị `v*`).

**(e) Lan truyền.** `best[r][c]` là cực đại, trên mọi đường (có thể không đơn) từ gốc tới ô, của `min`. Quy tắc `best[nb] = max(best[nb], min(best[cell], grid[nb]))` là phương trình tối ưu cho phép toán `(max, min)`; thuật toán dừng khi không còn cải thiện (giá trị tăng, bị chặn bởi các giá trị lưới) và khi đó thoả mọi phương trình ⇒ là nghiệm nhỏ nhất thoả ⇒ đúng.

**(f) Kiểm chứng bằng máy.** Mọi cách khớp vét cạn đường đơn trên mọi lưới nhỏ (mục 12) và ngẫu nhiên, và khớp nhau trên lưới `100 × 100`.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn đường đơn | mũ | `O(m n)` | oracle lưới nhỏ |
| 2. Nhị phân + BFS | `O(m n log(m n))` | `O(m n)` | dùng đơn điệu |
| 3. Max-heap Dijkstra | `O(m n log(m n))` | `O(m n)` | **đáp án chính** |
| 4. Union-Find | `O(m n log(m n))` | `O(m n)` | kích hoạt theo giá trị giảm dần |
| 5. Lan truyền | xấu `O((m n)²)` | `O(m n)` | đơn giản, thường nhanh |

Đo thực tế (`100 × 100`, giá trị ngẫu nhiên tới `10^9`):

In [17]:
def rand_grid(rng, m, n, vmax):
    return [[rng.randint(0, vmax) for _ in range(n)] for _ in range(m)]


def bench(fn, grid, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(grid)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(1102)
big = rand_grid(rng, 100, 100, 10 ** 9)
for name, fn in [("bsearch", max_min_bsearch), ("heap", max_min_heap), ("dsu", max_min_dsu), ("relax", max_min_relax)]:
    print(f"{name:8s}: {bench(fn, big) * 1000:8.1f} ms -> {fn(big)}")

bsearch :     38.2 ms -> 92111532


heap    :     10.4 ms -> 92111532
dsu     :     15.9 ms -> 92111532


relax   :     38.8 ms -> 92111532


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi lưới nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "bsearch": max_min_bsearch,
    "heap": max_min_heap,
    "dsu": max_min_dsu,
    "relax": max_min_relax,
    "Solution": lambda g: Solution().maximumMinimumPath([r[:] for r in g]),
    "SolutionBSearch": lambda g: SolutionBSearch().maximumMinimumPath([r[:] for r in g]),
    "SolutionHeap": lambda g: SolutionHeap().maximumMinimumPath([r[:] for r in g]),
    "SolutionDSU": lambda g: SolutionDSU().maximumMinimumPath([r[:] for r in g]),
    "SolutionRelax": lambda g: SolutionRelax().maximumMinimumPath([r[:] for r in g]),
}
BRUTE = {"brute": max_min_brute, "SolutionBrute": lambda g: SolutionBrute().maximumMinimumPath([r[:] for r in g])}


def check(grid, want=None, brute=True):
    if want is None:
        want = max_min_brute(grid) if brute else Solution().maximumMinimumPath([r[:] for r in grid])
    impls = dict(IMPLS)
    if brute:
        impls.update(BRUTE)
    for name, fn in impls.items():
        got = fn(grid)
        assert got == want, (name, grid, got, want)
    return want


for g, want in examples:
    check(g, want, brute=len(g) * len(g[0]) <= 12)
print("ví dụ đề: mọi cài đặt đúng;", len(IMPLS), "cách nhanh +", len(BRUTE), "vét cạn (lưới nhỏ)")

ví dụ đề: mọi cài đặt đúng; 9 cách nhanh + 2 vét cạn (lưới nhỏ)


### B. Edge case

- Lưới `1 × 1`: đáp án là chính ô đó.
- Lưới một hàng / một cột: đáp án là `min` của cả hàng/cột (chỉ có một đường).
- Lưới toàn bằng nhau.
- Điểm đầu hoặc điểm cuối là ô nhỏ nhất (đáp án bị chặn trên bởi hai góc).
- Có ô `0` ở giữa nhưng đường vòng tránh được; ô `0` bắt buộc phải qua.
- Đường tối ưu phải **đi lên/sang trái** (vòng tránh), không phải chỉ phải/xuống.
- Giá trị `10^9`.

In [19]:
edge = [
    ([[7]], 7), ([[3, 1, 4, 1, 5]], 1), ([[3], [9], [2]], 2), ([[5, 5], [5, 5]], 5),
    ([[1, 9], [9, 9]], 1), ([[9, 9], [9, 1]], 1), ([[9, 0, 9], [9, 0, 9], [9, 9, 9]], 9),
    ([[9, 0, 9], [9, 0, 9], [0, 9, 9]], 0), ([[8, 1, 8, 8], [8, 1, 1, 8], [8, 8, 8, 8]], 8),
    ([[10 ** 9, 10 ** 9], [0, 10 ** 9]], 10 ** 9),
    ([[9, 1, 9, 9, 9], [9, 1, 9, 1, 9], [9, 9, 9, 1, 9]], 9),
]
for g, want in edge:
    got = check(g, want, brute=len(g) * len(g[0]) <= 15)
print(len(edge), "edge case: mọi cài đặt khớp")
# lưới mà đường tối ưu BẮT BUỘC đi lên/trái (không thể chỉ phải/xuống):
snake = [[9, 1, 9, 9, 9], [9, 1, 9, 1, 9], [9, 9, 9, 1, 9]]
print("lưới ngoằn ngoèo:", Solution().maximumMinimumPath(snake))

11 edge case: mọi cài đặt khớp
lưới ngoằn ngoèo: 9


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_right_down_only(grid):                          # DP cổ điển: chỉ đi phải/xuống
    m, n = len(grid), len(grid[0])
    f = [[-1] * n for _ in range(m)]
    f[0][0] = grid[0][0]
    for r in range(m):
        for c in range(n):
            if (r, c) == (0, 0):
                continue
            up = f[r - 1][c] if r else -1
            left = f[r][c - 1] if c else -1
            f[r][c] = min(max(up, left), grid[r][c])
    return f[m - 1][n - 1]


def bug_greedy_walk(grid):                              # đi tham lam sang ô kề lớn nhất chưa thăm
    m, n = len(grid), len(grid[0])
    r = c = 0
    seen = {(0, 0)}
    ans = grid[0][0]
    while (r, c) != (m - 1, n - 1):
        cand = [(grid[r + dr][c + dc], r + dr, c + dc) for dr, dc in DIRS
                if 0 <= r + dr < m and 0 <= c + dc < n and (r + dr, c + dc) not in seen]
        if not cand:
            return -1
        v, r, c = max(cand)
        seen.add((r, c))
        ans = min(ans, v)
    return ans


def bug_min_of_corners(grid):                           # chỉ lấy min của hai góc
    return min(grid[0][0], grid[-1][-1])


def bug_exclude_endpoints(grid):                        # điểm không tính ô đầu và ô cuối
    m, n = len(grid), len(grid[0])
    if m * n <= 2:
        return grid[0][0]
    g2 = [row[:] for row in grid]
    big = 10 ** 18
    g2[0][0] = g2[m - 1][n - 1] = big
    return max_min_heap(g2)


def bug_min_heap(grid):                                 # dùng min-heap (lấy ô nhỏ nhất trước)
    m, n = len(grid), len(grid[0])
    heap = [(grid[0][0], 0, 0)]
    seen = [[False] * n for _ in range(m)]
    seen[0][0] = True
    ans = grid[0][0]
    while heap:
        v, r, c = heapq.heappop(heap)
        ans = min(ans, v)
        if (r, c) == (m - 1, n - 1):
            return ans
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc]:
                seen[nr][nc] = True
                heapq.heappush(heap, (grid[nr][nc], nr, nc))
    return ans


def bug_strict_reach(grid):                             # nhị phân với điều kiện '>' thay vì '>='
    def ok(v):
        m, n = len(grid), len(grid[0])
        if grid[0][0] <= v or grid[m - 1][n - 1] <= v:
            return False
        seen = {(0, 0)}
        st = [(0, 0)]
        while st:
            r, c = st.pop()
            if (r, c) == (m - 1, n - 1):
                return True
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen and grid[nr][nc] > v:
                    seen.add((nr, nc))
                    st.append((nr, nc))
        return False
    vals = sorted({v for row in grid for v in row})
    lo, hi = 0, len(vals) - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(vals[mid]):
            lo = mid
        else:
            hi = mid - 1
    return vals[lo]


BUGS = [bug_right_down_only, bug_greedy_walk, bug_min_of_corners, bug_exclude_endpoints, bug_min_heap, bug_strict_reach]
rng = random.Random(11020)
pool = [rand_grid(rng, rng.randint(1, 4), rng.randint(1, 4), 5) for _ in range(3000)]
# lưới ngoằn ngoèo (hai giá trị 0/9): đường tối ưu có thể BẮT BUỘC đi lên/trái; cần lưới rộng ít nhất 3 × 5
pool_up = [[[9 if rng.random() < 0.75 else 0 for _ in range(n)] for _ in range(m)] for m, n in [(3, 5), (4, 5), (3, 6)] * 150]
for g in pool_up:
    g[0][0] = g[-1][-1] = 9
for bug in BUGS:
    found = next((g for g in pool + pool_up if bug(g) != max_min_brute(g)), None)
    assert found, bug.__name__
    print(f"{bug.__name__:22s} sai trên {found}: ra {bug(found)}, đúng {max_min_brute(found)}")

bug_right_down_only    sai trên [[9, 9, 0, 9, 9], [9, 0, 9, 9, 9], [9, 9, 9, 0, 9]]: ra 0, đúng 9
bug_greedy_walk        sai trên [[3, 4, 5, 0], [3, 2, 0, 1], [5, 1, 2, 1], [3, 2, 0, 3]]: ra 0, đúng 1
bug_min_of_corners     sai trên [[3], [3], [0], [3]]: ra 3, đúng 0
bug_exclude_endpoints  sai trên [[0, 4], [5, 0], [5, 4], [0, 2]]: ra 4, đúng 0
bug_min_heap           sai trên [[3, 4, 5, 0], [3, 2, 0, 1], [5, 1, 2, 1], [3, 2, 0, 3]]: ra 0, đúng 1
bug_strict_reach       sai trên [[3, 4, 5, 0], [3, 2, 0, 1], [5, 1, 2, 1], [3, 2, 0, 3]]: ra 0, đúng 1


### D. Vét cạn mọi lưới nhỏ

Mọi lưới nhị phân/ba giá trị: `2 × 2` giá trị `0..3` (`256`), `2 × 3` và `3 × 2` giá trị `0..2` (`729` mỗi loại), `2 × 4` giá trị `0..2` (`6 561`), `3 × 3` giá trị `0..2` (`19 683`). So mọi cài đặt với vét cạn đường đơn.

In [21]:
t0 = time.perf_counter()
cnt = 0
for m, n, vmax in [(1, 1, 3), (1, 4, 2), (2, 2, 3), (2, 3, 2), (3, 2, 2), (2, 4, 2), (3, 3, 2)]:
    for vals in itertools.product(range(vmax + 1), repeat=m * n):
        g = [list(vals[i * n:(i + 1) * n]) for i in range(m)]
        want = max_min_brute(g)
        for name, fn in IMPLS.items():
            assert fn(g) == want, (name, g)
        cnt += 1
print(cnt, "lưới:", len(IMPLS), "cài đặt khớp vét cạn đường đơn;", round(time.perf_counter() - t0, 1), "s")

28043 lưới: 9 cài đặt khớp vét cạn đường đơn; 2.9 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Lưới ngẫu nhiên nhỏ (`≤ 4 × 4`, giá trị tới `10^9`): mọi cách khớp vét cạn; (2) lưới lớn: các cách khớp nhau; (3) **chuyển vị** và **xoay `180°`** không đổi đáp án; (4) **cộng hằng** cộng thêm vào đáp án; (5) **biến đổi tăng ngặt** `f` thì `ans(f(g)) = f(ans(g))`; (6) `min(grid) ≤ ans ≤ min(grid[0][0], grid[-1][-1])`; (7) **tăng một ô** không làm đáp án giảm; (8) đáp án là một giá trị có mặt trong lưới.

In [22]:
rng = random.Random(20240202)
t0 = time.perf_counter()
for it in range(600):
    m, n = rng.randint(1, 4), rng.randint(1, 4)
    g = rand_grid(rng, m, n, rng.choice([3, 10, 10 ** 9]))
    ans = check(g)                                                                    # (1)
    assert Solution().maximumMinimumPath([list(r) for r in zip(*g)]) == ans           # (3) chuyển vị
    assert Solution().maximumMinimumPath([r[::-1] for r in g[::-1]]) == ans           # (3) xoay 180
    c = rng.randint(0, 1000)
    assert Solution().maximumMinimumPath([[v + c for v in r] for r in g]) == ans + c  # (4)
    assert Solution().maximumMinimumPath([[v * 3 + 7 for v in r] for r in g]) == ans * 3 + 7   # (5)
    assert min(min(r) for r in g) <= ans <= min(g[0][0], g[-1][-1])                   # (6)
    r0, c0 = rng.randrange(m), rng.randrange(n)
    g2 = [r[:] for r in g]
    g2[r0][c0] += rng.randint(0, 5)
    assert Solution().maximumMinimumPath(g2) >= ans                                   # (7)
    assert ans in {v for r in g for v in r}                                           # (8)
print("(1)-(8) 600 lưới nhỏ:", round(time.perf_counter() - t0, 1), "s")

t0 = time.perf_counter()
for it in range(60):
    m, n = rng.randint(1, 40), rng.randint(1, 40)
    g = rand_grid(rng, m, n, rng.choice([5, 100, 10 ** 9]))
    ans = check(g, brute=False)                                                       # (2)
    assert Solution().maximumMinimumPath([list(r) for r in zip(*g)]) == ans
print("(2) 60 lưới tới 40 × 40: các cách khớp nhau;", round(time.perf_counter() - t0, 1), "s")

(1)-(8) 600 lưới nhỏ: 0.2 s


(2) 60 lưới tới 40 × 40: các cách khớp nhau; 0.4 s


### F. Giới hạn

`100 × 100`: ngẫu nhiên tới `10^9`; toàn bằng nhau; đường hầm "ngoằn ngoèo" buộc đi lên/trái; lưới có tường `0` với một lỗ; lưới tăng dần (từ góc trên trái xuống dưới phải); giá trị chỉ `0` và `10^9`. Các cách nhanh khớp nhau.

In [23]:
t0 = time.perf_counter()
rg = random.Random(21)
N = 100
cases = {}
cases["ngẫu nhiên"] = rand_grid(rg, N, N, 10 ** 9)
cases["hằng"] = [[7] * N for _ in range(N)]
cases["tăng dần"] = [[r + c for c in range(N)] for r in range(N)]
cases["chỉ 0 và 1e9"] = [[10 ** 9 if rg.random() < 0.7 else 0 for _ in range(N)] for _ in range(N)]
cases["chỉ 0 và 1e9"][0][0] = cases["chỉ 0 và 1e9"][N - 1][N - 1] = 10 ** 9
wall = [[9] * N for _ in range(N)]
for r in range(N):
    wall[r][50] = 0
wall[37][50] = 9                                          # một lỗ trên tường
cases["tường + lỗ"] = wall
snake = [[9] * N for _ in range(N)]
for r in range(1, N, 2):
    for c in range(N - 1):
        snake[r][c] = 0                                   # hàng lẻ là tường (chỉ chừa cột cuối)
cases["ngoằn ngoèo"] = snake
for name, g in cases.items():
    want = Solution().maximumMinimumPath([r[:] for r in g])
    for iname, fn in IMPLS.items():
        if iname.startswith("Solution") and iname != "Solution":
            continue
        assert fn(g) == want, (iname, name)
    print(f"  {name:14s}: {want}")
assert Solution().maximumMinimumPath(cases["hằng"]) == 7 and Solution().maximumMinimumPath(cases["tường + lỗ"]) == 9
assert Solution().maximumMinimumPath(cases["tăng dần"]) == 0
print("100 × 100:", round(time.perf_counter() - t0, 1), "s")

  ngẫu nhiên    : 177113153
  hằng          : 7


  tăng dần      : 0
  chỉ 0 và 1e9  : 1000000000
  tường + lỗ    : 9


  ngoằn ngoèo   : 9
100 × 100: 0.4 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ đi phải/xuống (DP cổ điển)

In [24]:
g = [[9, 0, 9, 9, 9], [9, 0, 9, 0, 9], [9, 9, 9, 0, 9]]
print("bug:", bug_right_down_only(g), "| đúng:", Solution().maximumMinimumPath(g))
assert bug_right_down_only(g) == 0 and Solution().maximumMinimumPath(g) == 9
bad = next(g for g in pool_up if bug_right_down_only(g) != max_min_brute(g))
print("phản ví dụ:", bad, "| bug:", bug_right_down_only(bad), "| đúng:", max_min_brute(bad))
assert bug_right_down_only(bad) < max_min_brute(bad)

bug: 0 | đúng: 9


phản ví dụ: [[9, 9, 0, 9, 9], [9, 0, 9, 9, 9], [9, 9, 9, 0, 9]] | bug: 0 | đúng: 9


**Vì sao sai:** đề cho phép **bốn hướng** nên đường tối ưu có thể đi **lên hoặc sang trái** để vòng tránh ô nhỏ. DP phải/xuống bỏ sót các đường đó.

### Lỗi 2: tham lam đi sang ô kề lớn nhất

In [25]:
bad = next(g for g in pool if bug_greedy_walk(g) != max_min_brute(g))
print("phản ví dụ:", bad, "| bug:", bug_greedy_walk(bad), "| đúng:", max_min_brute(bad))
assert bug_greedy_walk(bad) != max_min_brute(bad)

phản ví dụ: [[3, 4, 5, 0], [3, 2, 0, 1], [5, 1, 2, 1], [3, 2, 0, 3]] | bug: 0 | đúng: 1


**Vì sao sai:** đi theo ô lớn nhất trước mắt có thể dẫn vào ngõ cụt hoặc bỏ lỡ đường tốt hơn. Max-heap **mở rộng toàn bộ biên** (không đi một đường), nên không bỏ sót.

### Lỗi 3: chỉ lấy `min` của hai góc

In [26]:
bad = next(g for g in pool if bug_min_of_corners(g) != max_min_brute(g))
print("phản ví dụ:", bad, "| bug:", bug_min_of_corners(bad), "| đúng:", max_min_brute(bad))
assert bug_min_of_corners(bad) > max_min_brute(bad)

phản ví dụ: [[3], [3], [0], [3]] | bug: 3 | đúng: 0


**Vì sao sai:** hai góc chỉ cho **chặn trên**. Đường đi còn phải qua các ô trung gian có thể nhỏ hơn.

### Lỗi 4: không tính ô đầu và ô cuối

In [27]:
bad = next(g for g in pool if bug_exclude_endpoints(g) != max_min_brute(g))
print("phản ví dụ:", bad, "| bug:", bug_exclude_endpoints(bad), "| đúng:", max_min_brute(bad))
assert bug_exclude_endpoints(bad) != max_min_brute(bad)

phản ví dụ: [[0, 4], [5, 0], [5, 4], [0, 2]] | bug: 4 | đúng: 0


**Vì sao sai:** điểm là `min` của **mọi ô trên đường, gồm cả ô đầu và ô cuối**; `1 × 1` thì đáp án là chính ô đó.

### Lỗi 5: dùng min-heap

In [28]:
bad = next(g for g in pool if bug_min_heap(g) != max_min_brute(g))
print("phản ví dụ:", bad, "| bug:", bug_min_heap(bad), "| đúng:", max_min_brute(bad))
assert bug_min_heap(bad) < max_min_brute(bad)

phản ví dụ: [[3, 4, 5, 0], [3, 2, 0, 1], [5, 1, 2, 1], [3, 2, 0, 3]] | bug: 0 | đúng: 1


**Vì sao sai:** muốn **tối đa hoá** điểm cổ chai nên phải lấy ô **lớn nhất** trong biên trước. Min-heap lấy ô nhỏ nhất, kéo `ans` xuống một cách không cần thiết.

### Lỗi 6 (bonus): điều kiện `>` thay cho `>=` khi nhị phân

In [29]:
bad = next(g for g in pool if bug_strict_reach(g) != max_min_brute(g))
print("phản ví dụ:", bad, "| bug:", bug_strict_reach(bad), "| đúng:", max_min_brute(bad))
assert bug_strict_reach(bad) != max_min_brute(bad)

phản ví dụ: [[3, 4, 5, 0], [3, 2, 0, 1], [5, 1, 2, 1], [3, 2, 0, 3]] | bug: 0 | đúng: 1


**Vì sao sai:** đường có điểm `v` gồm các ô `≥ v` (**bằng** `v` vẫn được). Dùng `> v` loại luôn các đường có điểm đúng `v`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "đường đi tối đa hoá giá trị nhỏ nhất (hoặc tối thiểu hoá giá trị lớn nhất) trên đường" → **bài toán cổ chai (bottleneck path)** → **nhị phân + BFS**, **Dijkstra biến thể**, hoặc **Union-Find theo thứ tự giá trị**.

**Mẹo:**

1. `reach(v)` đơn điệu theo `v` ⇒ nhị phân trên đáp án.
2. Max-heap + `ans = min(ans, giá trị lấy ra)`; dừng khi lấy ra ô đích.
3. Union-Find: sắp xếp ô theo giá trị giảm dần, dừng khi hai góc nối được.
4. Cho phép bốn hướng: **không** dùng DP phải/xuống.
5. Điểm tính cả ô đầu và ô cuối.
6. Oracle: DFS mọi đường đơn trên lưới nhỏ.

**Họ bài cùng pattern:** Path With Minimum Effort (1631), Swim in Rising Water (778), The Earliest Moment When Everyone Become Friends (1101), Network Delay Time (743), Cheapest Flights Within K Stops (787), Minimum Cost to Make at Least One Valid Path in a Grid (1368).

## 15. Biến thể

1. **Khôi phục một đường** đạt điểm tốt nhất.
2. **Tối thiểu hoá giá trị lớn nhất** trên đường (minimax, kiểu Swim in Rising Water).
3. **Cho phép `8` hướng** (thêm đường chéo).
4. **Số ô** có thể tới được từ `(0,0)` với mọi ô `≥ v`.

Cài đặt và đối chiếu vét cạn:

In [30]:
def best_path(grid):
    m, n = len(grid), len(grid[0])
    heap = [(-grid[0][0], 0, 0)]
    parent = {(0, 0): None}
    ans = grid[0][0]
    while heap:
        neg, r, c = heapq.heappop(heap)
        ans = min(ans, -neg)
        if (r, c) == (m - 1, n - 1):
            path, cur = [], (r, c)
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            return ans, path[::-1]
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and (nr, nc) not in parent:
                parent[(nr, nc)] = (r, c)
                heapq.heappush(heap, (-grid[nr][nc], nr, nc))


def min_max_path(grid):
    m, n = len(grid), len(grid[0])
    heap = [(grid[0][0], 0, 0)]
    seen = {(0, 0)}
    ans = grid[0][0]
    while heap:
        v, r, c = heapq.heappop(heap)
        ans = max(ans, v)
        if (r, c) == (m - 1, n - 1):
            return ans
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen:
                seen.add((nr, nc))
                heapq.heappush(heap, (grid[nr][nc], nr, nc))


def brute_min_max(grid):
    m, n = len(grid), len(grid[0])
    best = [float("inf")]
    seen = [[False] * n for _ in range(m)]

    def dfs(r, c, cur):
        cur = max(cur, grid[r][c])
        if (r, c) == (m - 1, n - 1):
            best[0] = min(best[0], cur)
            return
        seen[r][c] = True
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc]:
                dfs(nr, nc, cur)
        seen[r][c] = False

    dfs(0, 0, -1)
    return best[0]


DIRS8 = [(dr, dc) for dr in (-1, 0, 1) for dc in (-1, 0, 1) if (dr, dc) != (0, 0)]


def max_min_8(grid):
    m, n = len(grid), len(grid[0])
    heap = [(-grid[0][0], 0, 0)]
    seen = {(0, 0)}
    ans = grid[0][0]
    while heap:
        neg, r, c = heapq.heappop(heap)
        ans = min(ans, -neg)
        if (r, c) == (m - 1, n - 1):
            return ans
        for dr, dc in DIRS8:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen:
                seen.add((nr, nc))
                heapq.heappush(heap, (-grid[nr][nc], nr, nc))


def count_reachable(grid, v):
    m, n = len(grid), len(grid[0])
    if grid[0][0] < v:
        return 0
    seen = {(0, 0)}
    dq = deque([(0, 0)])
    while dq:
        r, c = dq.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen and grid[nr][nc] >= v:
                seen.add((nr, nc))
                dq.append((nr, nc))
    return len(seen)


rng = random.Random(15)
for _ in range(500):
    g = rand_grid(rng, rng.randint(1, 4), rng.randint(1, 4), 6)
    ans, path = best_path(g)
    assert ans == max_min_brute(g) == score(g, path)
    assert path[0] == (0, 0) and path[-1] == (len(g) - 1, len(g[0]) - 1)
    assert all(abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1 for a, b in zip(path, path[1:]))
    assert min_max_path(g) == brute_min_max(g)
    assert max_min_8(g) == max_min_brute(g, DIRS8)
    v = rng.randint(0, 6)
    cnt = count_reachable(g, v)
    comp = {(0, 0)} if g[0][0] >= v else set()
    changed = True
    while changed:                                            # bao đóng liên thông độc lập bằng lặp tới điểm bất động
        changed = False
        for (r, c) in list(comp):
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < len(g) and 0 <= nc < len(g[0]) and g[nr][nc] >= v and (nr, nc) not in comp:
                    comp.add((nr, nc))
                    changed = True
    assert cnt == len(comp)
print("khôi phục đường / minimax / 8 hướng / đếm ô tới được: khớp vét cạn")

khôi phục đường / minimax / 8 hướng / đếm ô tới được: khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi đường đơn | mũ | `O(m n)` | oracle |
| 2. Nhị phân + BFS | `reach(v)` đơn điệu | `O(m n log(m n))` | `O(m n)` | đơn giản, nhớ dễ |
| 3. Max-heap | Dijkstra cổ chai | `O(m n log(m n))` | `O(m n)` | **đáp án chính** |
| 4. Union-Find | kích hoạt theo giá trị giảm | `O(m n log(m n))` | `O(m n)` | nhiều truy vấn ngưỡng |
| 5. Lan truyền | `best[r][c]` | xấu `O((m n)²)` | `O(m n)` | đơn giản |

Mọi cách khớp vét cạn trên `27 000+` lưới nhỏ và `600` lưới ngẫu nhiên; khớp nhau trên lưới `100 × 100`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [31]:
print(Solution().maximumMinimumPath([[3, 4], [5, 6]]), Solution().maximumMinimumPath([[9, 1, 9], [9, 1, 9], [9, 9, 9]]), Solution().maximumMinimumPath([[2, 8, 8], [1, 1, 8], [8, 8, 8]]), Solution().maximumMinimumPath([[7]]))

3 9 2 7


### Bài 2 (tính tay)

`grid = [[6,2,7],[9,3,5],[8,4,6]]`. Liệt kê `reach(v)` cho `v = 2, 3, 4, 5, 6` (hai góc có thông qua các ô `≥ v` không?) và suy ra đáp án. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện max-heap:

In [32]:
def max_min_todo(grid):
    m, n = len(grid), len(grid[0])
    heap = [(-grid[0][0], 0, 0)]
    seen = [[False] * n for _ in range(m)]
    seen[0][0] = True
    ans = grid[0][0]
    while heap:
        neg, r, c = heapq.heappop(heap)
        pass            # TODO 1: ans = min(ans, -neg); nếu là ô đích thì trả ans
        for dr, dc in DIRS:
            pass        # TODO 2: nếu ô kề hợp lệ và chưa thăm: đánh dấu và đẩy (-giá trị, nr, nc)
    return ans


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(max_min_todo(g) == w for g, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **Union-Find** (kích hoạt ô theo giá trị giảm dần):

In [33]:
class MySolution:
    def maximumMinimumPath(self, grid):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().maximumMinimumPath(g) == w for g, w in examples)

### Bài 5 (mini challenge): tối thiểu hoá giá trị lớn nhất trên đường

Viết `min_of_max(grid)` trả về **giá trị lớn nhất nhỏ nhất** trên một đường từ `(0,0)` đến `(m-1,n-1)` (4 hướng): tức `min_P max_{ô ∈ P} grid`. Ví dụ `min_of_max([[0,2],[1,3]]) == 3` và `min_of_max([[4]]) == 4` (bài kiểm tra ở mục 19 còn so với vét cạn). Gợi ý: đổi max-heap thành min-heap và `min` thành `max`.

In [34]:
def min_of_max(grid):
    pass            # TODO


# Khi làm xong:
# assert min_of_max([[0, 2], [1, 3]]) == 3 and min_of_max([[4]]) == 4

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: đáp án bị chặn trên bởi `min(hai góc)`.
- Mức 2: `[[3,4],[5,6]]`: hai đường `3→4→6` (`3`) và `3→5→6` (`3`) vì ô đầu là `3`.
- Mức 3: `3 9 2 7`: `[[9,1,9],[9,1,9],[9,9,9]]` đi vòng hàng dưới ⇒ `9`; `[[2,8,8],[1,1,8],[8,8,8]]` ô đầu là `2` ⇒ `2`; `[[7]]` ⇒ `7`.

**Bài 2**

- Mức 1: hai góc `6` và `6`; đáp án `≤ 6`.
- Mức 2: ô `≥ 6`: `6 @ (0,0)`, `7 @ (0,2)`, `9 @ (1,0)`, `8 @ (2,0)`, `6 @ (2,2)`; từ `(0,0)` xuống `(1,0)=9`, `(2,0)=8`, nhưng `(2,1)=4 < 6` chặn `(2,2)`.
- Mức 3: `reach(5)` sai: từ `(0,0)=6` chỉ đi được xuống `(1,0)=9`, `(2,0)=8` (các ô `≥ 5`), còn `(2,1)=4`, `(1,1)=3`, `(0,1)=2` đều `< 5` nên không tới được đích. `reach(4)` đúng nhờ đường `6→9→8→4→6`. Đáp án `4`.

**Bài 3**

- Mức 1: `ans = min(ans, -neg)`; `if (r, c) == (m - 1, n - 1): return ans`.
- Mức 2: kiểm tra `0 <= nr < m and 0 <= nc < n and not seen[nr][nc]`.
- Mức 3: `seen[nr][nc] = True`; `heapq.heappush(heap, (-grid[nr][nc], nr, nc))`.

**Bài 4**

- Mức 1: sắp xếp `(giá trị, r, c)` giảm dần.
- Mức 2: kích hoạt ô, hợp nhất với hàng xóm đã kích hoạt.
- Mức 3: khi `find(0) == find(m*n - 1)` (cả hai đã kích hoạt) trả giá trị ô vừa kích hoạt.

**Bài 5**

- Mức 1: min-heap theo giá trị; `ans = max(ans, v)`.
- Mức 2: dừng khi lấy ra ô đích.
- Mức 3: ví dụ `[[0,2],[1,3]]`: ô cuối là `3` nên đáp án `≥ 3`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [35]:
vals = (Solution().maximumMinimumPath([[3, 4], [5, 6]]), Solution().maximumMinimumPath([[9, 1, 9], [9, 1, 9], [9, 9, 9]]), Solution().maximumMinimumPath([[2, 8, 8], [1, 1, 8], [8, 8, 8]]), Solution().maximumMinimumPath([[7]]))
print(vals)
assert vals == (3, 9, 2, 7)
assert vals == tuple(max_min_brute(g) for g in ([[3, 4], [5, 6]], [[9, 1, 9], [9, 1, 9], [9, 9, 9]], [[2, 8, 8], [1, 1, 8], [8, 8, 8]], [[7]]))
print("Bài 1 OK")

(3, 9, 2, 7)
Bài 1 OK


### Bài 2

In [36]:
g2 = [[6, 2, 7], [9, 3, 5], [8, 4, 6]]
table = {v: reachable(g2, v) for v in (2, 3, 4, 5, 6)}
print(table)
assert table == {2: True, 3: True, 4: True, 5: False, 6: False}
assert Solution().maximumMinimumPath(g2) == 4 == max_min_brute(g2)
print("Bài 2 OK")

{2: True, 3: True, 4: True, 5: False, 6: False}
Bài 2 OK


### Bài 3

In [37]:
def max_min_answer(grid):
    m, n = len(grid), len(grid[0])
    heap = [(-grid[0][0], 0, 0)]
    seen = [[False] * n for _ in range(m)]
    seen[0][0] = True
    ans = grid[0][0]
    while heap:
        neg, r, c = heapq.heappop(heap)
        ans = min(ans, -neg)
        if (r, c) == (m - 1, n - 1):
            return ans
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc]:
                seen[nr][nc] = True
                heapq.heappush(heap, (-grid[nr][nc], nr, nc))
    return ans


assert all(max_min_answer(g) == w for g, w in examples)
for g in pool:
    assert max_min_answer(g) == max_min_brute(g)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def maximumMinimumPath(self, grid):
        m, n = len(grid), len(grid[0])
        parent = list(range(m * n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        active = [[False] * n for _ in range(m)]
        for v, r, c in sorted(((grid[r][c], r, c) for r in range(m) for c in range(n)), reverse=True):
            active[r][c] = True
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and active[nr][nc]:
                    parent[find(r * n + c)] = find(nr * n + nc)
            if active[0][0] and active[m - 1][n - 1] and find(0) == find(m * n - 1):
                return v


assert all(MySolutionAnswer().maximumMinimumPath(g) == w for g, w in examples)
for g in pool:
    assert MySolutionAnswer().maximumMinimumPath(g) == max_min_brute(g)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [39]:
def min_of_max(grid):
    return min_max_path(grid)


assert min_of_max([[0, 2], [1, 3]]) == 3 and min_of_max([[4]]) == 4
assert min_of_max([[0, 1, 2], [5, 5, 5], [1, 1, 0]]) == brute_min_max([[0, 1, 2], [5, 5, 5], [1, 1, 0]])
for g in pool:
    assert min_of_max(g) == brute_min_max(g)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đường đi cổ chai: tối đa hoá giá trị nhỏ nhất trên đường, 4 hướng):**

```python
class Solution:
    def maximumMinimumPath(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        heap = [(-grid[0][0], 0, 0)]
        seen = [[False] * n for _ in range(m)]
        seen[0][0] = True
        ans = grid[0][0]
        while heap:
            neg, r, c = heapq.heappop(heap)
            ans = min(ans, -neg)
            if r == m - 1 and c == n - 1:
                return ans
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc]:
                    seen[nr][nc] = True
                    heapq.heappush(heap, (-grid[nr][nc], nr, nc))
        return ans
```

**Checklist trước khi submit:**

- [ ] Được đi **4 hướng** (không dùng DP chỉ phải/xuống)?
- [ ] Điểm tính **cả ô đầu và ô cuối** (`ans` khởi tạo `grid[0][0]`)?
- [ ] Max-heap (đảo dấu) để lấy ô **lớn nhất** trong biên trước?
- [ ] Đánh dấu `seen` khi **đẩy vào heap** (tránh đẩy trùng) và dừng khi lấy ra ô đích?
- [ ] Nếu nhị phân: điều kiện `>= v`, `mid = (lo + hi + 1) // 2`?
- [ ] Đã thử `1 × 1`, một hàng, một cột, đường vòng buộc đi lên/trái?
- [ ] Đúng tên `class Solution`, `maximumMinimumPath(self, grid)`, không in debug.